<a href="https://colab.research.google.com/github/asegura4488/FisicaA/blob/main/Semana3/AlgoritmoEncuentro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# =========================================
# Parámetros físicos del problema
# =========================================
g = 9.81  # gravedad (m/s^2)

# Pelota 1
y1_0 = 0.0
v1_0 = 40.0

# Pelota 2
y2_0 = 90.0
v2_0 = 10.0

# =========================================
# Parámetros numéricos
# =========================================
dt = 0.01
t_total = 8.0
N = int(t_total / dt) + 1

# =========================================
# Arreglos
# =========================================
t = np.zeros(N)

y1 = np.zeros(N)
v1 = np.zeros(N)

y2 = np.zeros(N)
v2 = np.zeros(N)

# Condiciones iniciales
t[0] = 0.0

y1[0] = y1_0
v1[0] = v1_0

y2[0] = y2_0
v2[0] = v2_0

# =========================================
# Variables para detección del encuentro
# =========================================
encuentro_encontrado = False

t_aprox = None
y_aprox = None
v1_aprox = None
v2_aprox = None

t_interp = None
y_interp = None
v1_interp = None
v2_interp = None

# =========================================
# Integración por Euler + detección de cruce
# =========================================
for n in range(N - 1):
    # Avance temporal
    t[n + 1] = t[n] + dt

    # Euler para pelota 1
    y1[n + 1] = y1[n] + dt * v1[n]
    v1[n + 1] = v1[n] - g * dt

    # Euler para pelota 2
    y2[n + 1] = y2[n] + dt * v2[n]
    v2[n + 1] = v2[n] - g * dt

    # Diferencia de posiciones antes y después del paso
    f_n = y1[n] - y2[n]
    f_np1 = y1[n + 1] - y2[n + 1]

    # Detectar cruce exacto en un nodo
    if f_n == 0.0:
        encuentro_encontrado = True

        t_aprox = t[n]
        y_aprox = y1[n]
        v1_aprox = v1[n]
        v2_aprox = v2[n]

        t_interp = t[n]
        y_interp = y1[n]
        v1_interp = v1[n]
        v2_interp = v2[n]
        break

    # Detectar cambio de signo: el cruce ocurrió entre n y n+1
    if f_n * f_np1 < 0:
        encuentro_encontrado = True

        # ---------------------------------
        # Aproximación simple
        # Tomamos el punto n+1
        # ---------------------------------
        t_aprox = t[n + 1]
        y_aprox = y1[n + 1]
        v1_aprox = v1[n + 1]
        v2_aprox = v2[n + 1]

        # ---------------------------------
        # Interpolación lineal
        # ---------------------------------
        alpha = abs(f_n) / (abs(f_n) + abs(f_np1))

        t_interp = t[n] + alpha * dt
        y_interp = y1[n] + alpha * (y1[n + 1] - y1[n])
        v1_interp = v1[n] + alpha * (v1[n + 1] - v1[n])
        v2_interp = v2[n] + alpha * (v2[n + 1] - v2[n])

        break

# =========================================
# Solución exacta para comparar
# =========================================
t_exact = (y2_0 - y1_0) / (v1_0 - v2_0)
y_exact = y1_0 + v1_0 * t_exact - 0.5 * g * t_exact**2
v1_exact = v1_0 - g * t_exact
v2_exact = v2_0 - g * t_exact

# =========================================
# Impresión de resultados
# =========================================
print("=" * 50)
print("RESULTADOS DEL ENCUENTRO")
print("=" * 50)

print("\nSolución exacta:")
print(f"t_exacto   = {t_exact:.6f} s")
print(f"y_exacta   = {y_exact:.6f} m")
print(f"v1_exacta  = {v1_exact:.6f} m/s")
print(f"v2_exacta  = {v2_exact:.6f} m/s")

if encuentro_encontrado:
    print("\nAproximación simple (primer punto después del cruce):")
    print(f"t_aprox    = {t_aprox:.6f} s")
    print(f"y_aprox    = {y_aprox:.6f} m")
    print(f"v1_aprox   = {v1_aprox:.6f} m/s")
    print(f"v2_aprox   = {v2_aprox:.6f} m/s")

    print("\nInterpolación lineal entre dos pasos:")
    print(f"t_interp   = {t_interp:.6f} s")
    print(f"y_interp   = {y_interp:.6f} m")
    print(f"v1_interp  = {v1_interp:.6f} m/s")
    print(f"v2_interp  = {v2_interp:.6f} m/s")

    print("\nErrores absolutos:")
    print(f"|t_interp - t_exacto| = {abs(t_interp - t_exact):.6e} s")
    print(f"|y_interp - y_exacta| = {abs(y_interp - y_exact):.6e} m")
else:
    print("\nNo se detectó encuentro en el intervalo de simulación.")

RESULTADOS DEL ENCUENTRO

Solución exacta:
t_exacto   = 3.000000 s
y_exacta   = 75.855000 m
v1_exacta  = 10.570000 m/s
v2_exacta  = -19.430000 m/s

Aproximación simple (primer punto después del cruce):
t_aprox    = 3.010000 s
y_aprox    = 76.107850 m
v1_aprox   = 10.471900 m/s
v2_aprox   = -19.528100 m/s

Interpolación lineal entre dos pasos:
t_interp   = 3.000000 s
y_interp   = 76.002150 m
v1_interp  = 10.570000 m/s
v2_interp  = -19.430000 m/s

Errores absolutos:
|t_interp - t_exacto| = 1.287859e-14 s
|y_interp - y_exacta| = 1.471500e-01 m
